# True Signal SNR Calculation

Compute the true SNR of the injected signal using the optimal SNR formula:

$$\text{SNR} = \sqrt{4 \int_0^\infty \frac{|\tilde{h}(f)|^2}{S_n(f)} df} = \sqrt{4 \Delta f \sum_i \frac{|\tilde{h}(f_i)|^2}{S_n(f_i)}}$$

where $\tilde{h}(f)$ is the Fourier transform of the signal, $S_n(f)$ is the noise PSD, and $\Delta f = 1/T$ is the frequency resolution.

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from gwpy.timeseries import TimeSeries
from scipy import signal
import seaborn as sns

sns.set(style='whitegrid', context='paper', font_scale=1.5)

/home/x-ctirapongpra/.local/lib/python3.9/site-packages/gwpy/time/__init__.py:36: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  from lal import LIGOTimeGPS


## Configuration

In [2]:
# Detector configuration (same as preprocessing)
DETECTORS = {
    'H1': {
        'channel_name': 'H1:GW-H',
        't0_gps': 1126259462.423,
        'F_plus': 0.578742411175002,
        'F_cross': -0.45094782109531206,
        'color': 'blue'
    },
    'L1': {
        'channel_name': 'L1:GW-L',
        't0_gps': 1126259462.4160156,
        'F_plus': -0.5274334329518102,
        'F_cross': 0.20520960891727422,
        'color': 'orange'
    }
}

# Experiment parameters
DURATIONS = [4.0, 8.0, 16.0, 32.0]  # seconds
AMPLIFY_FACTORS = [1.25, 0.62, 0.31]

# Processing parameters (same as preprocessing)
scale_factor = 1e-23
target_sample_rate = 4096  # Hz
tukey_alpha = 0.1

# Paths
data_dir = '/home/x-ctirapongpra/scratch/gw-collapsar/h(t)'
output_base_dir = '../Frames_1000'

print(f"Configuration loaded")
print(f"Detectors: {list(DETECTORS.keys())}")
print(f"Durations: {DURATIONS}")
print(f"Amplitude factors: {AMPLIFY_FACTORS}")

Configuration loaded
Detectors: ['H1', 'L1']
Durations: [4.0, 8.0, 16.0, 32.0]
Amplitude factors: [1.25, 0.62, 0.31]


## Load Raw Signal Data

In [3]:
# Load raw strain polarizations
time_data = np.loadtxt(os.path.join(data_dir, 't.txt'), delimiter=',')
hp_x = np.loadtxt(os.path.join(data_dir, 'hp_x.txt'))
hx_x = np.loadtxt(os.path.join(data_dir, 'hx_x.txt'))

# Calculate sample rate
dt = time_data[1] - time_data[0]
sample_rate = 1.0 / dt

print(f"Raw data loaded:")
print(f"  Sample rate: {sample_rate:.2f} Hz")
print(f"  Duration: {time_data[-1] - time_data[0]:.3f} s")
print(f"  Samples: {len(time_data)}")

Raw data loaded:
  Sample rate: 10000.00 Hz
  Duration: 24.000 s
  Samples: 240000


## SNR Computation Function

In [4]:
def compute_signal_snr(signal_timeseries, psd_freqs, psd_values):
    """
    Compute the optimal matched-filter SNR of a signal.
    
    SNR = sqrt( 4 * Delta_f * Sum_i |d(f_i)|^2 / S(f_i) )
    
    Parameters
    ----------
    signal_timeseries : TimeSeries
        Time-domain signal data
    psd_freqs : np.ndarray
        PSD frequency array
    psd_values : np.ndarray
        PSD values at each frequency
    
    Returns
    -------
    float
        Optimal SNR
    """
    # Get signal data and parameters
    signal_data = signal_timeseries.value
    sample_rate = signal_timeseries.sample_rate.value
    duration = signal_timeseries.duration.value
    
    # Frequency resolution
    delta_f = 1.0 / duration
    delta_t = 1.0 / sample_rate
    
    # Compute FFT of signal (with proper units: multiply by delta_t)
    signal_fft = np.fft.rfft(signal_data) * delta_t
    signal_freqs = np.fft.rfftfreq(len(signal_data), d=delta_t)
    
    # Interpolate PSD to match signal frequency bins
    psd_interp = np.interp(signal_freqs, psd_freqs, psd_values, 
                           left=psd_values[0], right=psd_values[-1])
    
    # Avoid division by zero
    psd_interp = np.where(psd_interp > 0, psd_interp, np.inf)
    
    # Compute SNR^2
    # The factor of 4 comes from: 4 * integral from 0 to inf
    # delta_f converts the sum to an integral approximation
    integrand = np.abs(signal_fft)**2 / psd_interp
    snr_squared = 4.0 * delta_f * np.sum(integrand[1:])  # Skip DC component
    
    snr = np.sqrt(snr_squared)
    
    return snr

## Compute SNR for All Configurations

SNR scales linearly with amplitude: $\text{SNR}(A) = A \times \text{SNR}(1)$

Computing SNR for all combinations of duration (4s, 8s, 16s, 32s) and amplitude factors (1.25, 0.62, 0.31).

In [5]:
# Compute SNR for each duration and amplitude combination
# SNR scales linearly with amplitude: SNR(A) = A * SNR(1.0)
snr_results = {}
taper_duration = 0.5  # seconds, fixed taper duration from LIGO paper

for target_duration in DURATIONS:
    crop_duration = target_duration / 2.0
    config_key_base = f"dur_{int(target_duration):02d}s"
    
    print(f"\nDuration: {int(target_duration)}s")
    
    for amp_factor in AMPLIFY_FACTORS:
        config_key = f"{config_key_base}_amp_{amp_factor:.2f}"
        snr_results[config_key] = {}
        
        print(f"  Amplitude factor: {amp_factor:.2f}")
        
        for det_name, det_params in DETECTORS.items():
            # Generate signal with the specified amplitude
            h_strain = det_params['F_plus'] * hp_x + det_params['F_cross'] * hx_x
            
            # Create TimeSeries
            n_samples = len(h_strain)
            duration_raw = (n_samples - 1) * dt
            t_start = det_params['t0_gps'] - duration_raw / 2.0
            
            ts_raw = TimeSeries(
                h_strain,
                sample_rate=sample_rate,
                t0=t_start,
                name=det_params['channel_name'],
                unit='strain'
            )
            
            # Apply same processing as data generation
            t0_gps = det_params['t0_gps']
            
            # 1. Scale and crop (symmetric around t0_gps) - USE ACTUAL AMPLITUDE FACTOR
            ts_scaled = ts_raw * scale_factor * amp_factor
            ts_cropped = ts_scaled.crop(t0_gps - crop_duration/2, t0_gps + crop_duration/2)
            
            # 2. Resample to 4096 Hz
            ts_resampled = ts_cropped.resample(target_sample_rate)
            
            # 3. Apply Tukey taper BEFORE padding (to taper actual signal, not zeros)
            n_samples = len(ts_resampled)
            taper_samples = taper_duration * target_sample_rate
            tukey_alpha = min(2.0 * taper_samples / n_samples, 1.0)  # cap at 1.0
            
            tukey_window = signal.windows.tukey(n_samples, alpha=tukey_alpha)
            ts_tapered = ts_resampled * tukey_window
            
            # 4. Pad to target duration
            target_samples = int(target_duration * target_sample_rate)
            current_samples = len(ts_tapered)
            
            if current_samples < target_samples:
                total_padding_samples = target_samples - current_samples
                pad_left = total_padding_samples // 2
                pad_right = total_padding_samples - pad_left
                ts_temp = ts_tapered.pad((pad_left, pad_right))
                
                t0_int = int(t0_gps)
                desired_t0 = t0_int - (target_duration / 2.0)
                shift_amount = desired_t0 - ts_temp.t0.value
                ts_temp.shift(shift_amount)
                ts_final = ts_temp
            else:
                ts_final = ts_tapered
            
            # Load PSD from the corresponding amplitude directory
            psd_file = f"{output_base_dir}/dur_{int(target_duration):02d}s_amp_{amp_factor:.2f}/{det_name}_psd.dat"
            psd_data = np.loadtxt(psd_file)
            psd_freqs = psd_data[:, 0]
            psd_values = psd_data[:, 1]
            
            # Compute SNR
            snr = compute_signal_snr(ts_final, psd_freqs, psd_values)
            snr_results[config_key][det_name] = snr
            
            print(f"    {det_name}: SNR = {snr:.2f}")

print("\n" + "=" * 70)
print("SNR computation complete!")


Duration: 4s
  Amplitude factor: 1.25
    H1: SNR = 91.95
    L1: SNR = 91.27
  Amplitude factor: 0.62
    H1: SNR = 91.95
    L1: SNR = 91.27
  Amplitude factor: 0.31
    H1: SNR = 91.95
    L1: SNR = 91.27

Duration: 8s
  Amplitude factor: 1.25
    H1: SNR = 189.87
    L1: SNR = 189.43
  Amplitude factor: 0.62
    H1: SNR = 189.87
    L1: SNR = 189.43
  Amplitude factor: 0.31
    H1: SNR = 189.87
    L1: SNR = 189.43

Duration: 16s
  Amplitude factor: 1.25
    H1: SNR = 387.11
    L1: SNR = 387.84
  Amplitude factor: 0.62
    H1: SNR = 387.11
    L1: SNR = 387.84
  Amplitude factor: 0.31
    H1: SNR = 387.11
    L1: SNR = 387.84

Duration: 32s
  Amplitude factor: 1.25
    H1: SNR = 944.18
    L1: SNR = 956.80
  Amplitude factor: 0.62
    H1: SNR = 944.18
    L1: SNR = 956.80
  Amplitude factor: 0.31
    H1: SNR = 944.18
    L1: SNR = 956.80

SNR computation complete!


## Visualize SNR Results

In [6]:
# Display SNR summary table for all duration and amplitude combinations
print(f"\n{'Configuration':<20} {'H1 SNR':<12} {'L1 SNR':<12} {'Network SNR':<12}")
print("-" * 70)

for dur in DURATIONS:
    for amp in AMPLIFY_FACTORS:
        config_key = f"dur_{int(dur):02d}s_amp_{amp:.2f}"
        h1_snr = snr_results[config_key]['H1']
        l1_snr = snr_results[config_key]['L1']
        network_snr = np.sqrt(h1_snr**2 + l1_snr**2)  # quadrature sum for network SNR
        
        print(f"{int(dur)}s, amp={amp:.2f}{'':<4} {h1_snr:<12.2f} {l1_snr:<12.2f} {network_snr:<12.2f}")
    print()  # blank line between durations


Configuration        H1 SNR       L1 SNR       Network SNR 
----------------------------------------------------------------------
4s, amp=1.25     91.95        91.27        129.56      
4s, amp=0.62     91.95        91.27        129.56      
4s, amp=0.31     91.95        91.27        129.56      

8s, amp=1.25     189.87       189.43       268.20      
8s, amp=0.62     189.87       189.43       268.20      
8s, amp=0.31     189.87       189.43       268.20      

16s, amp=1.25     387.11       387.84       547.97      
16s, amp=0.62     387.11       387.84       547.97      
16s, amp=0.31     387.11       387.84       547.97      

32s, amp=1.25     944.18       956.80       1344.23     
32s, amp=0.62     944.18       956.80       1344.23     
32s, amp=0.31     944.18       956.80       1344.23     

